## Extract text from pdf files

In [ ]:
import fitz  # pymupdf
import os

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

def parse_filename(filename):
    name = filename.replace(".pdf", "")
    
    # split by "-"
    parts = name.split("-")
    
    return {
        "keywords": parts[0].replace("_", " "),
        "author": parts[1],
        "journal": parts[2],
        "year": parts[3]
    }

def load_all_papers(folder):
    documents = []
    for filename in os.listdir(folder):
        if filename.endswith(".pdf"):
            path = os.path.join(folder, filename)
            text = extract_text_from_pdf(path)
            metadata = parse_filename(filename)
            documents.append({
                "text": text,
                "filename": filename,
                "keywords": metadata["keywords"],
                "author": metadata["author"],
                "journal": metadata["journal"],
                "year": metadata["year"]
            })
    return documents


In [10]:
documents = load_all_papers(folder = "papers_pdf")

## Chunk the text

In [11]:
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap
    return chunks

def chunk_documents(documents, chunk_size=1000, overlap=200):
    chunks = []
    for doc in documents:
        doc_chunks = chunk_text(doc["text"], chunk_size, overlap)
        for i, chunk in enumerate(doc_chunks):
            chunks.append({
                "text": chunk,
                "chunk_id": i,
                "filename": doc["filename"],
                "author": doc["author"],
                "journal": doc["journal"],
                "year": doc["year"],
                "keywords": doc["keywords"]
            })
    return chunks

chunks = chunk_documents(documents)
print(f"Total chunks: {len(chunks)}")

Total chunks: 1612


## Build index for text search, vector search 

In [ ]:
def build_index(documents):
        index = Index(
        text_fields=['text'],
        keyword_fields=['filename','author','journal','year','keywords']
    )
    index.fit(documents)
    return index


def build_vindex(documents):
        index = Index(
        text_fields=['text'],
        keyword_fields=['filename','author','journal','year','keywords']
    )
    index.fit(documents)
    return index

In [13]:
import sqlitesearch as sq

TypeError: 'module' object is not callable